# AI Knowledge Assistant — RAG Demo

This notebook demonstrates the **end-to-end Retrieval-Augmented Generation (RAG)** pipeline
of the `ai_knowledge_assistant` project.

### Pipeline Overview

```mermaid
flowchart LR
    A["📁 Raw Files"] --> B["📖 Ingestion"]
    B --> C["✂️ Chunking"]
    C --> D["🔢 Embedding"]
    D --> E["💾 Storage"]
    E --> F["🔍 Retrieval"]
    F --> G["🚦 Grounding Gate"]
    G --> H["🤖 LLM Generation"]
    H --> I["✅ Citation Validation"]
    I --> J["📝 Grounded Answer"]
```

**Sections 1–6** run locally (no API key needed).  
**Sections 7–8** require an `OPENAI_API_KEY` environment variable.

## 1. Setup & Configuration

Import all pipeline modules and display the active configuration.

In [ ]:
import ai_knowledge_assistant.config as config
from ai_knowledge_assistant.embedding import (
    EmbeddingBuilder,
)
from ai_knowledge_assistant.grounding import AnswerabilityGate, OutputValidator
from ai_knowledge_assistant.ingest import read_files
from ai_knowledge_assistant.llm import LLMConfig, OpenAIClient
from ai_knowledge_assistant.normalize import get_chunks_from_files
from ai_knowledge_assistant.rag import build_prompt
from ai_knowledge_assistant.retrieve import Retriever
from ai_knowledge_assistant.store import save_chunks, save_chunks_vectors

print("=== Configuration ===")
print(f"Embedding model : {config.DEFAULT_EMBEDDING.model_name}")
print(f"Embedding dim   : {config.DEFAULT_EMBEDDING.dimension}")
print(f"Chunk size      : {config.CHUNK_SIZE}")
print(f"Chunk overlap   : {config.CHUNK_OVERLAP}")
print(f"Min total chars : {config.MIN_TOTAL_CHARS}")
print(f"LLM model       : {config.DEFAULT_LLM_MODEL}")
print(f"Index file      : {config.INDEX_FILE}")
print(f"Vectors file    : {config.VECTORS_FILE}")

=== Configuration ===
Embedding model : all-MiniLM-L6-v2
Embedding dim   : 384
Chunk size      : 500
Chunk overlap   : 100
Min total chars : 800
LLM model       : gpt-4.1
Index file      : data_set\output\index.json
Vectors file    : data_set\output\vectors.json


In [ ]:
import os
import sys

# Ensure the src directory is on the path
src_dir = os.path.join(os.getcwd(), "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

## 2. Ingestion — Reading Files

The `read_files` function recursively discovers `.txt` and `.md` files
and returns their content as a `dict[str, str]` mapping file path → text.

In [2]:
INPUT_DIR = os.path.join("data_set", "inputs")

text_by_file = read_files([INPUT_DIR])

print(f"Files discovered: {len(text_by_file)}")
print(f"Total characters: {sum(len(t) for t in text_by_file.values()):,}")
print()
for path, content in text_by_file.items():
    print(f"  {path}  ({len(content):,} chars)")

Files discovered: 9
Total characters: 36,928

  c:\Projects\ai_knowledge_assistant\data_set\inputs\a.txt  (1,679 chars)
  c:\Projects\ai_knowledge_assistant\data_set\inputs\b.txt  (1,638 chars)
  c:\Projects\ai_knowledge_assistant\data_set\inputs\c.txt  (17,410 chars)
  c:\Projects\ai_knowledge_assistant\data_set\inputs\dir2\ai_and_ml_overview.txt  (743 chars)
  c:\Projects\ai_knowledge_assistant\data_set\inputs\dir2\cooking_pasta.txt  (208 chars)
  c:\Projects\ai_knowledge_assistant\data_set\inputs\dir2\database_and_Storage.txt  (715 chars)
  c:\Projects\ai_knowledge_assistant\data_set\inputs\dir2\python_systems.txt  (780 chars)
  c:\Projects\ai_knowledge_assistant\data_set\inputs\t1\d.txt  (7,166 chars)
  c:\Projects\ai_knowledge_assistant\data_set\inputs\t1\e.txt  (6,589 chars)


## 3. Chunking — Splitting Text into Overlapping Chunks

Text is split into fixed-size chunks with overlap to preserve context
across chunk boundaries.  
Default: **500 chars** per chunk, **100 chars** overlap.

In [3]:
chunks = get_chunks_from_files(text_by_file)

print(f"Total chunks: {len(chunks)}")
print()

# Show the first 3 chunks as examples
for chunk in chunks[:3]:
    preview = chunk.text[:120].replace("\n", " ")
    print(f'  [{chunk.source}, chunk {chunk.index}] "{preview}..."')
    print(f"    Length: {len(chunk.text)} chars")
    print()

Total chunks: 96

  [c:\Projects\ai_knowledge_assistant\data_set\inputs\a.txt, chunk 0] "Cats are fascinating and beloved companions known for their independent and graceful nature. These domesticated felines ..."
    Length: 500 chars

  [c:\Projects\ai_knowledge_assistant\data_set\inputs\a.txt, chunk 1] "hough some breeds can be larger. Their eyes are adapted for excellent night vision, and their hearing is exceptionally a..."
    Length: 500 chars

  [c:\Projects\ai_knowledge_assistant\data_set\inputs\a.txt, chunk 2] "ous vocalizations including meowing, purring, and hissing, as well as through body language and scent marking.  Breeds a..."
    Length: 500 chars



## 4. Embedding — Generating Vectors

Each chunk is encoded into a dense vector using `sentence-transformers`
(model: `all-MiniLM-L6-v2`, dimension: 384).  
This runs **locally** — no API key required.

In [4]:
builder = EmbeddingBuilder(config=config.DEFAULT_EMBEDDING)
embedded_chunks = builder.build_vectors(chunks)

print(f"Embedded {len(embedded_chunks)} chunks")
print(f"Vector dimension: {len(embedded_chunks[0].vector)}")
print()

# Preview the first vector
sample = embedded_chunks[0]
print(f"Sample — [{sample.source}, chunk {sample.index}]")
print(f"  Vector (first 10 values): {[round(v, 4) for v in sample.vector[:10]]}")

c:\Projects\ai_knowledge_assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedded 96 chunks
Vector dimension: 384

Sample — [c:\Projects\ai_knowledge_assistant\data_set\inputs\a.txt, chunk 0]
  Vector (first 10 values): [0.091, -0.0182, 0.0331, 0.098, -0.1009, 0.0054, 0.0088, -0.0374, -0.0747, -0.0006]


## 5. Storage — Persisting Index and Vectors

Chunks are saved to `index.json` and vectors to `vectors.json`.
These files are later loaded for retrieval.

In [5]:
os.makedirs(os.path.dirname(config.INDEX_FILE), exist_ok=True)

save_chunks(chunks, path=config.INDEX_FILE)
save_chunks_vectors(embedded_chunks, path=config.VECTORS_FILE)

index_size = os.path.getsize(config.INDEX_FILE)
vectors_size = os.path.getsize(config.VECTORS_FILE)

print(f"Saved {config.INDEX_FILE}   ({index_size:,} bytes)")
print(f"Saved {config.VECTORS_FILE} ({vectors_size:,} bytes)")

Saved data_set\output\index.json   (59,021 bytes)
Saved data_set\output\vectors.json (1,128,440 bytes)


## 6. Retrieval — Semantic Search

The `Retriever` loads the vectors, builds an in-memory FAISS index,
encodes the question, and returns the most relevant chunks
ranked by **cosine similarity**.

In [6]:
retriever = Retriever(
    embedding_config=config.DEFAULT_EMBEDDING,
    vectors_path=config.VECTORS_FILE,
)

DEMO_QUESTION = "What role does Python play in machine learning and large systems?"

results = retriever.retrieve(DEMO_QUESTION, limit=5)

print(f"Question: {DEMO_QUESTION}")
print(f"Retrieved {len(results)} context chunks:\n")
for chunk in results:
    preview = chunk.text[:150].replace("\n", " ")
    print(f"  [{chunk.source}, chunk {chunk.index}]")
    print(f'    "{preview}..."')
    print()

Question: What role does Python play in machine learning and large systems?
Retrieved 4 context chunks:

  [c:\Projects\ai_knowledge_assistant\data_set\inputs\dir2\ai_and_ml_overview.txt, chunk 0]
    "Artificial intelligence is a broad field concerned with building intelligent systems. Machine learning is a subfield that focuses on learning patterns..."

  [c:\Projects\ai_knowledge_assistant\data_set\inputs\dir2\ai_and_ml_overview.txt, chunk 1]
    "arning research. This is due to frameworks such as PyTorch and TensorFlow. Researchers use Python to experiment with models quickly.  In production AI..."

  [c:\Projects\ai_knowledge_assistant\data_set\inputs\dir2\python_systems.txt, chunk 0]
    "Python is widely used in modern software systems. It is often chosen for backend services, automation, and scripting. One reason for its popularity is..."

  [c:\Projects\ai_knowledge_assistant\data_set\inputs\dir2\python_systems.txt, chunk 1]
    " engineering pipelines. It is frequently used t

## 7. Grounding — Answerability Gate & Prompt

Before calling the LLM, the **Answerability Gate** checks whether
the retrieved context is sufficient (≥ 800 total characters).  
If it passes, we build the grounded prompt with source citations.

In [7]:
# Answerability check
gate = AnswerabilityGate()
total_chars = sum(len(c.text) for c in results)
is_answerable = gate.should_answer(results)

print(f"Total context chars : {total_chars:,}")
print(f"Min required        : {config.MIN_TOTAL_CHARS}")
print(f"Answerability gate  : {'✅ PASS' if is_answerable else '❌ FAIL'}")
print()

# Build grounded prompt
prompt = build_prompt(DEMO_QUESTION, results)
print("=== Generated Prompt (first 600 chars) ===")
print(prompt[:600])
print("...")

Total context chars : 1,723
Min required        : 800
Answerability gate  : ✅ PASS

=== Generated Prompt (first 600 chars) ===
You are an AI assistant answering questions based strictly on provided context.

You must follow these rules:
1. Use ONLY the provided context.
2. Do NOT use prior knowledge.
3. For every factual statement, cite the source in this exact format: [Source: <filename>, chunk <index>]
4. If the answer cannot be found in the context, respond exactly: INSUFFICIENT_CONTEXT

--- CONTEXT ---

[Source: ai_and_ml_overview.txt, chunk 0]
Artificial intelligence is a broad field concerned with building intelligent systems.
Machine learning is a subfield that focuses on learning patterns from data.

In machin
...


## 8. LLM Generation & Citation Validation

> **Requires `OPENAI_API_KEY`** — set it in your environment before running.

The prompt is sent to OpenAI. After generation, the **Output Validator**
ensures every `[Source: file, chunk N]` citation references a chunk
that was actually retrieved.

In [8]:
if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️  OPENAI_API_KEY not set — skipping LLM generation.")
    print("Set the variable and re-run this cell to see the full demo.")
else:
    llm = OpenAIClient()
    llm_config = LLMConfig(model=config.DEFAULT_LLM_MODEL)

    answer = llm.generate(prompt, config=llm_config)

    # Citation validation
    validator = OutputValidator()
    is_valid = validator.validate(answer=answer, contexts=results)

    print("=== ANSWER ===")
    print(answer)
    print()
    print(f"Citation validation: {'✅ PASS' if is_valid else '❌ FAIL'}")

=== ANSWER ===
Python plays a significant role in machine learning by serving as the dominant language for research, largely due to frameworks such as PyTorch and TensorFlow. Researchers use Python to experiment with models quickly. In production AI systems, Python is often used in supporting components such as monitoring, scaling, and data validation [Source: ai_and_ml_overview.txt, chunk 1].

In large systems, Python is commonly used to glue components together, integrating well with C++ modules and external services, making it suitable for orchestration layers. It is also popular in data engineering pipelines for processing logs, transforming datasets, and validating data. Even when performance-critical parts are written in other languages, Python often remains the control layer that coordinates execution, a design pattern common in cloud-based architectures [Source: python_systems.txt, chunk 0; Source: python_systems.txt, chunk 1].

Citation validation: ❌ FAIL


---

## Pipeline Summary

| Step | Module | What happened |
|------|--------|---------------|
| **Ingestion** | `ingest.read_files` | Discovered and read all `.txt`/`.md` files |
| **Chunking** | `normalize.get_chunks_from_files` | Split text into overlapping 500-char chunks |
| **Embedding** | `embedding.EmbeddingBuilder` | Encoded chunks into 384-dim vectors (local model) |
| **Storage** | `store.save_chunks` / `save_chunks_vectors` | Persisted index and vectors to JSON |
| **Retrieval** | `retrieve.Retriever` | Semantic search via FAISS (cosine similarity) |
| **Grounding** | `grounding.AnswerabilityGate` | Verified sufficient context before LLM call |
| **Generation** | `llm.OpenAIClient` | Generated answer via OpenAI with grounded prompt |
| **Validation** | `grounding.OutputValidator` | Verified all citations reference retrieved chunks |